In [ ]:
!unzip -q dataset_xy.zip

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms
import glob
import PIL.Image
import os
import numpy as np

# --- 1. DATASET CLASS ---
# This reads the xy_XXX_YYY filenames and normalizes them for the AI
class XYDataset(torch.utils.data.Dataset):
    def __init__(self, directory, transform=None):
        self.directory = directory
        self.image_paths = glob.glob(os.path.join(directory, '*.jpg'))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = PIL.Image.open(image_path)

        # Parse x, y from filename: xy_XXX_YYY_UUID.jpg
        filename = os.path.basename(image_path)
        items = filename.split('_')
        x = float(int(items[1]))
        y = float(int(items[2]))

        # Normalize to [-1, 1] (Standard for JetBot Regression)
        x = 2.0 * (x / 224.0) - 1.0
        y = 2.0 * (y / 224.0) - 1.0

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor([x, y]).float()

# --- 2. PREPARE DATA ---
# We use ColorJitter to help the robot handle different lighting conditions
trans = transforms.Compose([
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = XYDataset('dataset_xy', transform=trans)

# Split: 90% for training, 10% for testing
train_size = len(dataset) - int(len(dataset) * 0.1)
test_size = len(dataset) - train_size
train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = torch.utils.data.DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=16, shuffle=True)

# --- 3. THE MODEL (ResNet-18) ---
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, 2) # Outputting X and Y coordinates
device = torch.device('cuda') # Ensure Colab Runtime is set to GPU
model = model.to(device)

# --- 4. THE TRAINING LOOP ---
NUM_EPOCHS = 200
BEST_MODEL_PATH = 'best_steering_model_xy.pth'
best_loss = 1e9
optimizer = optim.Adam(model.parameters())

print("Starting training...")
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    for images, labels in iter(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.mse_loss(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += float(loss)

    # Validation
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for images, labels in iter(test_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_loss += float(F.mse_loss(outputs, labels))

    avg_train_loss = train_loss / len(train_loader)
    avg_test_loss = test_loss / len(test_loader)

    print(f'Epoch {epoch}: Train Loss {avg_train_loss:.4f}, Test Loss {avg_test_loss:.4f}')

    if avg_test_loss < best_loss:
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_loss = avg_test_loss
        print("Model saved!")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 38.2MB/s]


Starting training...


/tmp/ipython-input-869362235.py:85: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  train_loss += float(loss)


Epoch 0: Train Loss 1.0478, Test Loss 2.6620
Model saved!
Epoch 1: Train Loss 0.3093, Test Loss 9.5920
Epoch 2: Train Loss 0.0550, Test Loss 2.4873
Model saved!
Epoch 3: Train Loss 0.0319, Test Loss 0.5578
Model saved!
Epoch 4: Train Loss 0.0214, Test Loss 0.0165
Model saved!
Epoch 5: Train Loss 0.0101, Test Loss 0.0158
Model saved!
Epoch 6: Train Loss 0.0063, Test Loss 0.0314
Epoch 7: Train Loss 0.0046, Test Loss 0.0282
Epoch 8: Train Loss 0.0056, Test Loss 0.0058
Model saved!
Epoch 9: Train Loss 0.0071, Test Loss 0.0034
Model saved!
Epoch 10: Train Loss 0.0055, Test Loss 0.0056
Epoch 11: Train Loss 0.0038, Test Loss 0.0032
Model saved!
Epoch 12: Train Loss 0.0053, Test Loss 0.0047
Epoch 13: Train Loss 0.0035, Test Loss 0.0010
Model saved!
Epoch 14: Train Loss 0.0025, Test Loss 0.0027
Epoch 15: Train Loss 0.0036, Test Loss 0.0058
Epoch 16: Train Loss 0.0045, Test Loss 0.0018
Epoch 17: Train Loss 0.0018, Test Loss 0.0023
Epoch 18: Train Loss 0.0025, Test Loss 0.0021
Epoch 19: Train Los

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>